In [2]:
import lseg.data as ld
import pandas as pd


In [5]:
ld.open_session()

<lseg.data.session.Definition object at 0x150e4c890 {name='workspace'}>

In [3]:
from src.modelling import TICKERS
print(TICKERS)

['NVDA.O', 'AMD.O', 'TSM.N', 'KO.N', 'PEP.O', 'JPM.N', 'BAC.N', 'GS.N', 'MS.N', 'XOM.N', 'CVX.N', 'AMZN.O', 'MSFT.O', 'META.O', 'GOOGL.O', 'JNJ.N', 'PFE.N']


In [ ]:
df = ld.get_history(
    ['US3MT=RR'],
    start="2019-01-01",
    end="2023-12-31",
)

yield_df = df["A_YLD_1"]
yield_df

Date
2019-01-02    2.4123
2019-01-03    2.3967
2019-01-04    2.4039
2019-01-07    2.3987
2019-01-08    2.4535
               ...  
2023-12-22     5.361
2023-12-26    5.3576
2023-12-27     5.391
2023-12-28    5.3641
2023-12-29    5.3323
Name: A_YLD_1, Length: 1275, dtype: Float64

In [12]:
import plotly.graph_objects as go

yield_df = yield_df.dropna()
fig = go.Figure()

fig.add_trace(go.Scatter(
    x=yield_df.index,
    y=yield_df,
    mode='lines',
    name='Risk-Free Rate (3M T-Bill)',
    line=dict(color='royalblue', width=1.5)
))

fig.update_layout(
    title='US 3-Month T-Bill Yield (Risk-Free Rate)',
    xaxis_title='Date',
    yaxis_title='Rate (%)',
    xaxis_rangeslider_visible=False,
    template='plotly_white',
    hovermode='x unified'
)

fig.show()

In [10]:
# rf_yield: pd.Series of A_YLD_1 in percent, indexed by date
rf_yield_clean = yield_df.dropna()

# 1) Convert to decimal annual yield
rf_annual = rf_yield_clean / 100.0

# 2) Sample average over 2019–2023
rf_annual_avg = rf_annual.mean()      # e.g. 0.025 → 2.5%

# 3) Round to 1 significant figure for the thesis
rf_annual_avg_sf = float(f"{rf_annual_avg:.1g}")   # 0.025 → 0.03 (3%)

# 4) Constant daily rate for Sharpe
rf_daily_const = rf_annual_avg / 252

print(rf_annual_avg_sf)

0.02


# Return Estimation & Data Pipeline Tests (TODO)

This notebook is for **manual smoke tests** of the Refinitiv data client and the
modelling/backtesting pipeline. Run it in your local environment where
`lseg.data` is configured.

## TODOs

- [ ] Verify that `get_close_prices` returns a tidy `DataFrame` with 1 column per RIC
- [ ] Confirm that historical close prices are accessible as `Series` for each ticker
- [ ] Run a small Engle–Granger test and spread analysis on real LSEG data
- [ ] Run the pairs backtest engine on real price series and inspect metrics
- [ ] Compare spread-based expected returns (from `estimate_expected_returns`) vs naive historical mean

The cells below contain suggested test code; adapt tickers and dates as needed.

In [ ]:
# --- 2. Test spread exploration / modelling functions on real data (optional) ---

from src.modelling.cointegration import screen_pairs
from src.modelling.spread_analysis import compute_spread, compute_zscore, spread_summary

pairs_to_test = [("NVDA.O", "AMD.O")]

try:
    # Reuse close_df from the previous cell if available; otherwise fetch again
    if "close_df" not in globals():
        close_df = get_close_prices([t for pair in pairs_to_test for t in pair], start=start, end=end)

    assert isinstance(close_df, pd.DataFrame)

    # cointegration on list of tickers (DataFrame input)
    coint_df = screen_pairs(close_df, pairs_to_test)
    display(coint_df)

    row = coint_df.iloc[0]
    y = close_df[row["y"]]
    x = close_df[row["x"]]

    spread = compute_spread(y, x, hedge_ratio=row["hedge_ratio"], intercept=row["intercept"])
    z = compute_zscore(spread, window=60)
    summary = spread_summary(y, x, hedge_ratio=row["hedge_ratio"], intercept=row["intercept"], window=60)

    print("Spread head:\n", spread.head())
    print("Z-score head:\n", z.head())
    print("Summary:", summary)

except Exception as exc:
    print("Spread / cointegration test failed:", exc)

## Notes

- The **Refinitiv tests** (cells 1–2) require a valid `lseg.data` configuration and
  credentials in your environment (e.g. via `.env`). In this environment they may
  raise authentication / network errors — that is expected.
- The **synthetic-data test** (cell 3) does **not** depend on LSEG at all and
  should run anywhere. It verifies that:
  - `screen_pairs` works end-to-end on a `DataFrame` of prices.
  - `PairsBacktestEngine.run` executes without errors on plausible data.
  - `estimate_expected_returns`, `mean_variance_weights`, and `efficient_frontier`
    all produce outputs with sensible shapes.

Use this notebook as a quick regression check when you modify the data,
modelling, or backtesting pipeline.